# Notebook 01 - Intro to Multimodal Systems

This notebook introduces the Castalia multimodal layer: the shift from text-only generation to systems that must perceive, ground, and justify answers across images, pages, audio, and video.


## What you will learn

- why multimodal work starts after text-only prompting, retrieval, and systems basics
- how to map modalities to concrete product workloads
- how to think about evidence, latency, and auditability before selecting a model


In [ ]:
!pip install -q "transformers>=4.57.0" accelerate torch numpy pandas matplotlib Pillow pydantic
print("Installed notebook dependencies.")


In [ ]:
import json
import math
import random
import statistics
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-multimodal")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
NOTEBOOK_ROOT = ARTIFACT_ROOT / "01_intro_to_multimodal_systems"
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(7)
np.random.seed(7)



print("Cache dir:", CACHE_DIR)
print("Artifact root:", NOTEBOOK_ROOT.resolve())


## Step 1: Put modalities next to real tasks

Strong multimodal design starts from workloads, not model demos. The table below keeps the task, required evidence, and operational risk in one place.


In [ ]:
workloads = pd.DataFrame([
    {"task": "invoice extraction", "modalities": "document,image", "core_evidence": "layout + table cells", "primary_risk": "ungrounded fields"},
    {"task": "chart question answering", "modalities": "image,text", "core_evidence": "bars, labels, axes", "primary_risk": "hallucinated trends"},
    {"task": "meeting notes", "modalities": "audio,text", "core_evidence": "transcript segments", "primary_risk": "timestamp drift"},
    {"task": "tutorial summary", "modalities": "video,audio,text", "core_evidence": "sampled frames + speech", "primary_risk": "missing events"},
    {"task": "screen debugging", "modalities": "image,text", "core_evidence": "UI widgets + logs", "primary_risk": "wrong affordance grounding"},
])
workloads


## Step 2: Budget the request before the model

Every multimodal request consumes a different mix of patches, tokens, audio seconds, and sampled frames. Budgeting those inputs early keeps the stack honest.


In [ ]:
budgets = pd.DataFrame([
    {"workload": "invoice extraction", "text_tokens": 800, "image_patches": 784, "audio_seconds": 0, "sampled_frames": 0},
    {"workload": "chart QA", "text_tokens": 250, "image_patches": 576, "audio_seconds": 0, "sampled_frames": 0},
    {"workload": "meeting recap", "text_tokens": 600, "image_patches": 0, "audio_seconds": 180, "sampled_frames": 0},
    {"workload": "video tutorial summary", "text_tokens": 500, "image_patches": 0, "audio_seconds": 90, "sampled_frames": 48},
])
budgets["relative_cost"] = (
    budgets["text_tokens"] / 1000
    + budgets["image_patches"] / 700
    + budgets["audio_seconds"] / 120
    + budgets["sampled_frames"] / 32
).round(2)
budgets.sort_values("relative_cost", ascending=False)


## Step 3: Decide when multimodal is actually necessary

The goal is not to make every task multimodal. The goal is to use more modalities only when the missing evidence materially changes the answer.


In [ ]:
decision_rules = pd.DataFrame([
    {"question": "Is layout essential?", "if_yes": "Use page or image pipeline", "if_no": "Stay text-first"},
    {"question": "Is sound itself meaningful?", "if_yes": "Use audio-text or audio embedding path", "if_no": "Transcribe first"},
    {"question": "Does the answer depend on temporal order?", "if_yes": "Use sampled-frame video path", "if_no": "Use still-frame extraction"},
    {"question": "Must the answer cite evidence?", "if_yes": "Store coordinates, timestamps, or frame spans", "if_no": "Free-form generation may be enough"},
])

def multimodal_needed(layout=False, sound=False, temporal=False):
    return sum([layout, sound, temporal]) >= 1

print("Need multimodal for chart QA:", multimodal_needed(layout=True))
print("Need multimodal for plain FAQ:", multimodal_needed())
decision_rules


## Exercises and extensions

1. Replace the toy workloads with three tasks from your own domain and recalculate the relative cost column.
1. Add a column that records what form of evidence should be saved for later audit: bounding box, page id, timestamp, or frame span.
